In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.model_selection import train_test_split


# Combine feature arrays into a single array and create labels for each case


target_configs = [
    {'presence_bt': 1, 'presence_bb': 0, 'presence_tb': 0, 'count_bt': 1, 'count_bb': 0, 'count_tb': 0},  # 1 pole on [bt]
    {'presence_bt': 0, 'presence_bb': 1, 'presence_tb': 0, 'count_bt': 0, 'count_bb': 1, 'count_tb': 0},  # 1 pole on [bb]
    {'presence_bt': 0, 'presence_bb': 0, 'presence_tb': 1, 'count_bt': 0, 'count_bb': 0, 'count_tb': 1},  # 1 pole on [tb]
    {'presence_bt': 1, 'presence_bb': 1, 'presence_tb': 0, 'count_bt': 1, 'count_bb': 1, 'count_tb': 0},  # 1 pole on [bt] and 1 pole on [bb]
    {'presence_bt': 0, 'presence_bb': 1, 'presence_tb': 1, 'count_bt': 0, 'count_bb': 1, 'count_tb': 1},  # 1 pole on [bb] and 1 pole on [tb]
    {'presence_bt': 1, 'presence_bb': 1, 'presence_tb': 1, 'count_bt': 1, 'count_bb': 1, 'count_tb': 1},  # 1 pole on each of [bt], [bb], and [tb]
    {'presence_bt': 0, 'presence_bb': 1, 'presence_tb': 1, 'count_bt': 0, 'count_bb': 2, 'count_tb': 1},  # 2 poles on [bb] and 1 pole on [tb]
    {'presence_bt': 0, 'presence_bb': 1, 'presence_tb': 1, 'count_bt': 0, 'count_bb': 1, 'count_tb': 2}   # 1 pole on [bb] and 2 poles on [tb]
]
# Repeat each configuration for all samples in each feature set
num_samples = [features_0.shape[0], features_1.shape[0], features_2.shape[0], features_3.shape[0],
               features_4.shape[0], features_5.shape[0], features_6.shape[0], features_7.shape[0]]

# Prepare target arrays
target_array = []

for i, config in enumerate(target_configs):
    # Create a row of 6 values for each sample in this feature set
    row = [config['presence_bt'], config['presence_bb'], config['presence_tb'],
           config['count_bt'], config['count_bb'], config['count_tb']]
    target_array.extend([row] * num_samples[i])

# Convert the target array into a torch tensor
targets = torch.tensor(target_array, dtype=torch.float32)

# Verify target shape


# Verify target sizes match number of samples
print("Feature array shape:", all_features.shape)
print("Target array shape:", targets.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class PoleDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = targets
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(all_features, targets, test_size=0.05, random_state=42)

# Create Dataset objects
train_dataset = PoleDataset(X_train, y_train)
test_dataset = PoleDataset(X_test, y_test)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import matplotlib.pyplot as plt
import torch.optim as optim
from torch import nn


class PoleClassifier(nn.Module):
    def __init__(self, input_size=100, hidden_size=100):
        super(PoleClassifier, self).__init__()
        # Define layers
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
        # Outputs for binary presence of poles on each position
        self.presence_bt = nn.Linear(hidden_size, 1)  # Binary presence for [bt]
        self.presence_bb = nn.Linear(hidden_size, 1)  # Binary presence for [bb]
        self.presence_tb = nn.Linear(hidden_size, 1)  # Binary presence for [tb]
        
        # Outputs for count of poles at each position
        self.count_bt = nn.Linear(hidden_size, 1)     # Count of poles on [bt]
        self.count_bb = nn.Linear(hidden_size, 1)     # Count of poles on [bb]
        self.count_tb = nn.Linear(hidden_size, 1)     # Count of poles on [tb]
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        
        # Presence predictions (binary)
        presence_bt = torch.sigmoid(self.presence_bt(x))
        presence_bb = torch.sigmoid(self.presence_bb(x))
        presence_tb = torch.sigmoid(self.presence_tb(x))
        
        # Count predictions (continuous)
        count_bt = F.relu(self.count_bt(x))
        count_bb = F.relu(self.count_bb(x))
        count_tb = F.relu(self.count_tb(x))
        
        return torch.concatenate([presence_bt,
                presence_bb,
                presence_tb,
                count_bt,
                count_bb,
                count_tb],dim=1)


# Custom loss function combining binary cross-entropy (for presence) and MSE (for counts)
class CustomLoss(nn.Module):
    def __init__(self):
        super(CustomLoss, self).__init__()
    
    def forward(self, outputs, targets):

        # Split outputs and targets into presence and count parts
        pred_presence = outputs[:, :3]  # Presence predictions (bt, bb, tb)
        pred_counts = outputs[:, 3:]    # Count predictions (bt, bb, tb)

        target_presence = targets[:, :3]  # Presence targets
        target_counts = targets[:, 3:]    # Count targets

        # Binary Cross-Entropy for presence
        presence_loss = nn.BCEWithLogitsLoss()(pred_presence, target_presence)
        
        # Mean Squared Error for counts
        count_loss = nn.MSELoss()(pred_counts, target_counts)

        # Total loss (combining both)
        total_loss = presence_loss + count_loss
        return total_loss

# Initialize the custom loss
criterion = CustomLoss()
model = PoleClassifier()

# Optimizer (e.g., Adam)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Track loss for plotting
train_losses = []
test_losses = []

# Number of epochs for training
num_epochs = 1000

# Loop through epochs
for epoch in range(num_epochs):
    # Train phase
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()  # Reset gradients
        outputs = model(inputs)  # Get model predictions
        loss = criterion(outputs, targets)  # Calculate custom loss
        loss.backward()  # Backpropagate the loss
        optimizer.step()  # Update the model weights
        
        train_loss += loss.item()  # Accumulate loss for tracking
    
    # Average training loss for this epoch
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Test phase (evaluate model on validation/test data)
    model.eval()
    test_loss = 0.0
    with torch.no_grad():  # No need to track gradients for evaluation
        for inputs, targets in test_loader:
            outputs = model(inputs)  # Get model predictions
            loss = criterion(outputs, targets)  # Calculate custom loss
            test_loss += loss.item()  # Accumulate loss for tracking
    
    # Average test loss for this epoch
    test_loss /= len(test_loader)
    test_losses.append(test_loss)
    
    # Print epoch stats
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

In [ ]:
# Plot train and test loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, 490 + 1), train_losses, label='Train Loss', color='blue')
plt.plot(range(1, 490 + 1), test_losses, label='Test Loss', color='red')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Test Loss over Epochs')
plt.legend()
plt.grid(True)

# plt.yscale("log")
plt.ylim(0,2)
plt.show()

In [ ]:
model.eval()
    
with torch.no_grad():  # No need to track gradients for evaluation
    for inputs, targets in test_loader:
        outputs = model(inputs)  # Get model predictions
        

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(5,5))

axs[0].imshow(targets)
axs[1].imshow(np.round(outputs.numpy() ))

fig.tight_layout()

In [ ]:
import torch

def get_class_from_output(outputs):
    # Round the outputs
    presence_bt, presence_bb, presence_tb = np.round(outputs[0].numpy()), np.round(outputs[1].numpy()), np.round(outputs[2].numpy())
    count_bt, count_bb, count_tb = np.round(outputs[3].numpy()), np.round(outputs[4].numpy()), np.round(outputs[5].numpy())

    # Map the rounded output to a class (0-7)
    class_mapping = {
        (1, 0, 0, 0, 0, 0): 0,  # Presence: bt, count: bt = 0
        (0, 1, 0, 0, 0, 0): 1,  # Presence: bb, count: bb = 0
        (0, 0, 1, 0, 0, 0): 2,  # Presence: tb, count: tb = 0
        (1, 1, 0, 1, 0, 0): 3,  # Presence: bt + bb, count: bt = 1
        (1, 0, 1, 0, 1, 0): 4,  # Presence: bt + tb, count: bb = 1
        (0, 1, 1, 0, 1, 0): 5,  # Presence: bb + tb, count: bb = 1
        (1, 1, 1, 1, 1, 0): 6,  # Presence: bt + bb + tb, count: bt = 1
        (1, 0, 0, 2, 0, 0): 7   # Presence: bt, count: bt = 2
    }
    
    # Map the rounded values to the class label
    class_label = class_mapping.get((presence_bt, presence_bb, presence_tb, count_bt, count_bb, count_tb), -1)

    print(presence_bt, presence_bb, presence_tb, count_bt, count_bb, count_tb, class_label)
    return class_label

def convert_to_class_matrix(outputs):
    """
    Given the outputs for a batch of samples, convert them to the class matrix
    where each row corresponds to the class of a sample.
    """
    class_labels = []

    for output in outputs:
        class_label = get_class_from_output(output)
        class_labels.append(class_label)
    
    return torch.tensor(class_labels)



class_matrix = convert_to_class_matrix(outputs)
print(class_matrix)  # This will give the class labels for each sample in the batch.


In [ ]:
def get_class_from_output(outputs):
    # Convert outputs into a tuple of presence and count
    presence_bt, presence_bb, presence_tb = outputs[:,0], outputs[:,1], outputs[:,2]
    count_bt, count_bb, count_tb = outputs[:,3], outputs[:,4], outputs[:,5]
    
    # Convert continuous count values into discrete values (e.g., rounding)
    count_bt = np.round(count_bt.numpy())
    count_bb = np.round(count_bb.numpy())
    count_tb = np.round(count_tb.numpy())
    
    # Convert presence to binary (0 or 1)
    presence_bt = np.round(presence_bt.numpy())
    presence_bb = np.round(presence_bb.numpy())
    presence_tb = np.round(presence_tb.numpy())
    
    # Create a tuple of (presence_bt, presence_bb, presence_tb, count_bt, count_bb, count_tb)
    # We map this tuple into a class index
    class_mapping = {
        (1, 0, 0, 1, 0, 0): 0,  # Class 0
        (0, 1, 0, 0, 1, 0): 1,  # Class 1
        (0, 0, 1, 0, 0, 1): 2,  # Class 2
        (1, 1, 0, 1, 1, 0): 3,  # Class 3
        (0, 1, 1, 0, 1, 1): 4,  # Class 4
        (1, 1, 1, 1, 1, 1): 5,  # Class 5
        (0, 2, 0, 0, 2, 0): 6,  # Class 6
        (0, 1, 2, 0, 1, 2): 7   # Class 7
    }
    
    return class_mapping.get((presence_bt, presence_bb, presence_tb, count_bt, count_bb, count_tb), -1)  # -1 if no match
get_class_from_output(outputs)